### Defining The Ising Model
We start with 2d lattice where each lattice site $i$ has a corresponing 'spin' $ \sigma_i \in \{-1, +1\} $. The microstate of lattice spins is notated by $\boldsymbol{\sigma}$. A Hamiltonian of a state $\boldsymbol{\sigma}$ in the Ising Model (in the absence of an external field) is defined by
$$H(\boldsymbol{\sigma}) = -\sum_{\langle i j \rangle} J_{ij} \sigma_i \sigma_j \tag{1}
$$
Where $\langle i j \rangle$ indicates $i$ and $j$ are adjacent lattice sites and $J_{ij}$ is the 'coupling strength' between given sites. In this experiment $J_{ij}$ is set equal to $1$ for all interactions and the Hamiltonian becomes simply
$$H(\boldsymbol{\sigma}) = -\sum_{\langle i j \rangle} \sigma_i \sigma_j \tag{2}
$$
The probability of any given state in a thermodynamic system in equilibrium at temperature $T$ coresponding to Bolzman factor $\beta = 1/(k_BT)$ is given by 
$$ p(\boldsymbol{\sigma}) = \frac{e^{-\beta H(\boldsymbol{\sigma})}}{Z} \tag{3}
 $$
where $Z_{\beta}$ is the partition function 
$$ Z = \sum_{\boldsymbol{\sigma}} e^{-\beta H(\boldsymbol{\sigma})}  \tag{4}
$$


### Computational Algorithms
Numerically simulating the Ising Model to produce representative statistics requires the above equations be satisfied aswell as that the simulation be in an equilibrium state. One method to ensure equilibrium is by ensuring condition of **detailed balance**. That is, that for any two microstates $\nu$ and $\mu$.
$$p(\mu) P(\mu \to \nu) =p(\nu) P(\nu \to \mu) \tag{5}
$$
#### The Metropolis Hastings Algorythm
The Metropolis Hastings Algorythm is one such algorythm that satisfies detailed balance by only changing one spin state $\sigma_i$ at a time (so for any $\nu$ and $\mu$ with more than one spin states difference $P(\mu \to \nu) = P(\nu \to \mu) = 0$ for a given simulation step). To Satisfy $(5)$, $P(\mu \to \nu)$ and $P(\nu \to \mu)$ must be chosen such that  
$$
\frac{P(\mu \to \nu)}{P(\nu \to \mu)} = \frac{p(\nu)}{p(\mu)} = \frac{\frac{e^{-\beta H(\nu)}}{Z}}{\frac{e^{-\beta H(\mu)}}{Z}} = e^{-\beta(H(\nu)-H(\mu))} \tag{6}
$$
This can be done by the following method:
Let $\nu$ be the current state
Choose a random lattice site and let $\mu$ be the proposed state of flipping the spin on the chosen lattice site, the probability to be computed is $P(\nu \to \mu)$ 
1) Choose a random lattice site 
2) Calculate $\Delta H = H(\nu)-H(\mu)$
3) - If $\Delta H > 0$,
set $P(\mu \to \nu) = 1$, then to satisfy $(6)$, let $P(\nu \to \mu) = e^{-\beta(\Delta H)}$ 
    - Else if $\Delta H < 0$,
by symetry of the above line, let $P(\nu \to \mu) = 1$  

These steps can be repeated and will produce true Ising Model statistics as the number of steps goes to infinity. This is one of the simplest ways to simulate the Ising Model

#### The Swendsen-Wang Algorythm
The Metropolis-Hastings algorythm produces true statistics as the number of steps goes to infinity, and produces sufficiently accurate statistics in a finite number of steps for temperatures above and bellow the critical temperature. However around the critical temperature the number of steps required to produce representative distributions goes to infitity, this phenomenon is known as **Critical Slowing Down**, see URL for a more in depth investigation. In order to study the system at criticality, for example, to find the **Critical Exponents**. An algorythm must be used that is computationally stable at and around the critical temperature. One algorythm which satisfies this requirement is the Swendsen-Wang Algorythm. Bellow we will show it also satisfies detailed balance for the Ising Model. 

As before, start with a state given by $\mu$. The Swendsen-Wang Algorythm works by finding many possible $\nu$ and tranforming to any found state $\nu$ with uniform probability. This works by introducing a new element, for each pair of adjacent lattice sites $i$ and $j$ assign a 'bond' variable $d_{i,j} \in \{0, 1\}$ these are assigned according to a probability distribution conditional on the spin states $\sigma_i$ and $\sigma_j$ contained in $\mu$. New states $\nu$ are then derived directly from the bond configuration $d$.

This can be thought of as splitting the probability $P(\nu \to \mu)$ into conditional probabilities $P(\nu \to \mu) = P(\mu \mid d)P(d \mid \nu)$. The procedure is as follows:

1) Bond Formation Step:
    for each bond variable $d_{i,j}$, set to 1 with the following probabilities
    $$
    P(d_{i,j} = 1) =
    \begin{cases} 
    1 - e^{-2\beta}, & \text{if }  \sigma_i = \sigma_j \\
    0, & \text{if } \sigma_i \neq \sigma_j
    \end{cases}
    $$

    $$
    P(d_{i,j} = 1 \mid \sigma_i = \sigma_j) = 1 - e^{-2\beta J}\\
    P(d_{i,j} = 0 \mid \sigma_i = \sigma_j) = e^{-2\beta J}\\
    P(d_{i,j} = 1 \mid \sigma_i \neq \sigma_j) = 0\\
    P(d_{i,j} = 0 \mid \sigma_i \neq \sigma_j) = 1
    $$
2) Clustering Step:
    Any two lattice sites $i$ and $j$ which are connected by a bond d_{i,j} = 1, must have the same spin in $\mu$.  Even if lattice sites are not adjacent, if there are bonds which connect them via intermediate lattice sites, the previous condition nececitates that they must have the same spin. In this way, clusters can be formed. A clustering algorythm can be used to complete this step, given a bond configuration $d$

3) Cluster flipping step:
    Given the space of possible $\mu$'s with each cluster being of one spin, a final $\mu$ can be picked uniformly by simply assigning each cluster either spin 1 or -1 with a probability of 1/2
$$
\begin{aligned}
P(d \mid \nu) = \prod_{\langle i j \rangle} & \Big[  P(d_{i,j}=1 \mid \sigma_i = \sigma_j) d_{i,j} (1 - \delta_{\sigma_i, \sigma_j})  \\
& + P(d_{i,j}=0 \mid \sigma_i = \sigma_j)(1 - d_{i,j})  \delta_{\sigma_i, \sigma_j} \\
& +  P(d_{i,j}=1 \mid \sigma_i \neq \sigma_j) d_{i,j} (1 - \delta_{\sigma_i, \sigma_j}) \\
& + P(d_{i,j}=0 \mid \sigma_i \neq \sigma_j)(1 - d_{i,j}) (1 - \delta_{\sigma_i, \sigma_j}) \Big]
\end{aligned}
$$


$$
P(d \mid \nu) = \prod_{\langle i j \rangle} \Bigg[
(1 - e^{-2\beta J}) d_{i,j} \delta_{\sigma_i, \sigma_j} 
+ e^{-2\beta J} (1 - d_{i,j}) \delta_{\sigma_i, \sigma_j} 
+ (1 - d_{i,j}) (1 - \delta_{\sigma_i, \sigma_j})
\Bigg]
$$

Applying the identity for a binary variable $x \in \{0, 1\}$

$$
ax + b(1-x) \equiv a^xb^{1-x}
$$
to $\delta_{\sigma_i, \sigma_j}$ and $d_{i,j}$

$$
P(d \mid \nu) = \prod_{\langle i j \rangle} \Bigg[
\left( (1 - e^{-2\beta J})^{d_{i,j}} e^{-2\beta J (1 - d_{i,j})} \right){\delta_{\sigma_i, \sigma_j}}+ (1 - d_{i,j}){(1 - \delta_{\sigma_i, \sigma_j})}
\Bigg].
$$

Consider each factor in the above product. If $\delta_{\sigma_i, \sigma_j}=1$ the right term becomes 0 and the left term becomes the factor. If $\delta_{\sigma_i, \sigma_j}=0$, the left term becomes 0 and the probability $P(d_{i,j} = 0 \mid \sigma_i \neq \sigma_j)$ = 1 enforces that $d_{i,j} = 0$ so the right term becomes $1$, contributing nothing to the overall product. For this reason we can neglect the right term and write

$$
P(d \mid \nu) = \prod_{\langle i j \rangle} \Bigg[
\left( (1 - e^{-2\beta J})^{d_{i,j}} e^{-2\beta J (1 - d_{i,j})} \right){\delta_{\sigma_i, \sigma_j}}
\Bigg].
$$









$$
\delta_{\sigma_i, \sigma_j} = (\sigma_i \sigma_j + 1)/2
$$

$$
P(d \mid \nu) = \prod_{\langle i j \rangle} \frac{1}{2} \Bigg[
(1 - e^{-2\beta J}) d_{i,j} (\sigma_i \sigma_j + 1) \\
+ e^{-2\beta J} (1 - d_{i,j}) (\sigma_i \sigma_j + 1) \\
+ (1 - d_{i,j})(1 - \sigma_i \sigma_j)
\Bigg]
$$

$$
P(d \mid \nu) = \sum_{\langle i j \rangle}{((2\sigma_i \sigma_j-1)/2)(1 - e^{-2\beta})} + \sum_{\langle i j \rangle}{((1-2\sigma_i \sigma_j)/2)(e^{-2\beta})}
$$


$$
\frac{P(\mu \to \nu)}{P(\nu \to \mu)} = \frac{P(\mu \mid d)P(d \mid \nu)}{P(\nu \mid d)P(d \mid \mu)} = \frac{P(d \mid \nu)}{P(d \mid \mu)} = e^{-\beta(H(\nu)-H(\mu))}
$$
